# Test `download_opentopography_dem`

Downloads a DEM covering a bounding box given in PSAD56 / UTM zone 19S (EPSG:24879), the package default CRS.

Bounding box (from a Leapfrog/geomodel bounding-box readout):

| Axis | Minimum   | Maximum   |
|------|-----------|-----------|
| X    | 511874    | 519187    |
| Y    | 7.57272e6 | 7.57768e6 |
| Z    | 3496.37   | 4198.56   |

Z is not used by `download_opentopography_dem` (it only needs the 2D footprint), but is shown here for reference.

In [1]:
from pathlib import Path

import sys

# This notebook lives at <repo>/notebooks/, so the package source is one level up in src/.
_SRC_DIR = Path(r"c:\Users\amsraguirre\OneDrive - Grupo Minero Antofagasta Minerals\01_REPOSITORIO_PROYECTOS\2026\geomodeltools\src")
if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(_SRC_DIR))

from geomodeltools.dem import download_opentopography_dem

ModuleNotFoundError: No module named 'geopandas'

In [8]:
# Bounding box in EPSG:24879 (PSAD56 / UTM zone 19S)
bounds = [511874, 7572720, 519187, 7576800]  # [minx, miny, maxx, maxy]
crs = "EPSG:24879"

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
out_tiff_path = out_dir / "dem_test_24879.tif"

In [ ]:
# Optional: add a margin (in meters) around the bounding box before requesting the DEM
margin_m = 500

dem_path = download_opentopography_dem(
    bounds=bounds,
    out_tiff_path=out_tiff_path,
    crs=crs,
    margin_m=margin_m,
    demtype="AW3D30",
    out_crs=crs,  # reproject the downloaded WGS84 DEM back to EPSG:24879
)
dem_path

## Inspect the resulting DEM

Check that the CRS is EPSG:24879 and that the raster bounds cover the requested bounding box (plus margin).

In [ ]:
import rasterio

with rasterio.open(dem_path) as src:
    print("CRS:", src.crs)
    print("Raster bounds:", src.bounds)
    print("Requested bounds (with margin):", [
        bounds[0] - margin_m, bounds[1] - margin_m,
        bounds[2] + margin_m, bounds[3] + margin_m,
    ])
    print("Shape:", src.width, "x", src.height)
    print("Nodata:", src.nodata)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with rasterio.open(dem_path) as src:
    band1 = src.read(1).astype(float)
    if src.nodata is not None:
        band1[band1 == src.nodata] = np.nan

plt.figure(figsize=(6, 6))
plt.imshow(band1, cmap="terrain")
plt.colorbar(label="Elevation (m)")
plt.title("Downloaded DEM (EPSG:24879)")
plt.show()